In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {'Baseline_Training', 'CTC_models', 'PhoneticFeatures_Training'}:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import (
    BASELINE_MODELS_DIR,
    CTC_MODELS_DIR,
    DEFAULT_BASELINE_MODEL,
    DEFAULT_CTC_ATTENTION_MODEL,
    DEFAULT_CTC_MODEL,
    DEFAULT_PHONETIC_MODEL_2CLASS,
    DEFAULT_PHONETIC_MODEL_3CLASS,
    EXAMPLE_EMBEDDINGS,
    EXAMPLE_PHONEMES,
    EXAMPLE_SEG_B2,
    EXAMPLE_SEG_B4,
    EXAMPLE_WAV,
    PHONEME_CHOICES_SUMMARY,
    PHONETIC_MODELS_DIR,
    TABLES_DIR,
)


In [ ]:
import torch
import os
import numpy as np
from content_manager.vencoder.HubertSoft import HubertSoft
from get_embeddings import process_directory_no_av, process_directory, process_audio_and_extract_embs_no_av, process_audio_and_extract_embs

In [ ]:
input_directory = r'\\nid-tts-02\mnt\hot_store\tts_data\CORPRES'
file_path = input_directory + '\Vladimir\Base\\a_corr\obmen\\ata1001-1500\\ata1101'
aims = ['get_all_non_av_embs', 'get_all_embs', 'get_selected_files', 'get_selected_files_no_av']
aim = aims[3]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
content_encoder = HubertSoft(device=device)



In [ ]:
if aim == 'get_all_non_av_embs':
    output_directory = r'\\nid-tts-02\mnt\hot_store\trainee_common\ananeva\CORPRES_embs_no_averaging'
    process_directory_no_av(input_directory, output_directory, content_encoder, device)

In [ ]:
if aim == 'get_all_embs':
    output_directory = r'\\nid-tts-02\mnt\hot_store\trainee_common\ananeva\CORPRES_embs'
    process_directory(input_directory, output_directory, content_encoder, device)

In [ ]:
if aim == 'get_selected_files':
    name = file_path.split('\\')[-1]
    output_directory = r'C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\Main_project\results'
    result = process_audio_and_extract_embs(input_directory, content_encoder, device)
    
    new_dict = {}
    for phoneme, tensor in result.items():
        new_dict[phoneme] = tensor.cpu().numpy()
    np.save(os.path.join(output_directory, name + "_embs.npy"), new_dict)

    with open(os.path.join(output_directory, name + "_phonemes.txt"), "w", encoding='utf-8') as f:
        for phoneme in result.keys():
            f.write(f"{phoneme.rstrip('_')}\n")
    

In [ ]:
if aim == 'get_selected_files_no_av':
    name = file_path.split('\\')[-1]
    output_directory = r'C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\Main_project\results'
    result = process_audio_and_extract_embs_no_av(file_path, content_encoder, device)

    keys = []
    values = []

    for ph in result:
        for key, value in ph.items():
            keys.append(key)
            value = value.cpu().numpy()
            values.append(value)

    with open(os.path.join(output_directory, name + '_no_av_ph.txt'), 'w', encoding='utf-8') as f:
        for phoneme in keys:
            f.write(f"{phoneme}\n")

    np.save(os.path.join(output_directory, name + "_no_av_embs.npy"), values)
    print('file saved')
